In [74]:
import sys
from pathlib import Path

import torch
from torchvision.transforms import v2 as transforms


this_path = Path(__file__) if '__file__' in globals() else Path("<undefined>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = work_path / Path("../torch-tools")
sys.path.append(str(tools_path))

ee_tools_path_p = work_path / Path("ee")
sys.path.append(str(ee_tools_path_p))

import utils
from datasets import fetch_handler
from run_manager import RunManager, RunsManager
from network import Network, Networks
from trainer import MultiTrainer, Trainer

from ee_tools.models.resnet_cifar_ee import resnet18 as resnet18_cifar_ee
from ee_tools.models.resnet_cifar_hase import resnet18 as resnet18_cifar_hase
from ee_tools.models.resnet_cifar_ee_cinit import resnet18 as resnet18_cifar_ee_cinit
from models_h.model_factories import create_model_ensembles
from models.resnet_cifar import resnet18 as resnet18_cifar

In [75]:
model_config = {
    "name": "resnet18",
    "num_classes": 100,
    "T": 1,
    "pretrained": False,
    "div": 1,  # is_ee=True の場合、この引数は無視されます
    "ensembles": 16,  # YAMLの 'num_models: 4' に対応
    "for_cifar_customize": True,
    "is_ee": True,
    "is_he": False
}

model = create_model_ensembles(**model_config)

net1 = Network(model)

check:   --- >  resnet18 100 1 False 1 16 True True


In [76]:
net0 = Network(resnet18_cifar_ee(num_classes=100, channels=16, ensembles=16))
net2 = Network(resnet18_cifar_hase(num_classes=100, channels=16, ensembles=16))
net3 = Network(resnet18_cifar_ee_cinit(num_classes=100, channels=16, ensembles=16))
net4 = Network(resnet18_cifar(num_classes=100))


In [77]:
utils.comp_param_stat([net0, net1, net2, net3, net4], layer_width=50)

Network: Network                                                            Network: Network                                                            Network: Network                                                            Network: Network                                                            Network: Network                                                        
------------------------------------------------------------------------    ------------------------------------------------------------------------    ------------------------------------------------------------------------    ------------------------------------------------------------------------    ------------------------------------------------------------------------
Layer                                                    Mean        Var    Layer                                                    Mean        Var    Layer                                                    Mean        Var    Layer             

In [78]:
print(net0.torchinfo(input_size=(1, 3, 32, 32)))

Layer (type (var_name))                  Input Shape               Output Shape              Kernel Shape              Param #                   Mult-Adds
ResNet (ResNet)                          [1, 3, 32, 32]            [1, 100]                  --                        --                        --
├─RepeatData (rp_data)                   [1, 3, 32, 32]            [1, 48, 32, 32]           --                        --                        --
├─Conv2d (conv1)                         [1, 48, 32, 32]           [1, 256, 32, 32]          [3, 3]                    6,912                     7,077,888
├─BatchNorm2d (bn1)                      [1, 256, 32, 32]          [1, 256, 32, 32]          --                        512                       512
├─ReLU (relu)                            [1, 256, 32, 32]          [1, 256, 32, 32]          --                        --                        --
├─Sequential (layer1)                    [1, 256, 32, 32]          [1, 256, 32, 32]          --  